In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base = "../data/"

out = pd.read_csv(base + "demo_output.csv")
ques = pd.read_csv(base + "demo_questions.csv")

print(ques.iloc[8]["question"])

# fragments = load_and_split_documents("data/knowledge-base.zip")
# found = search(question, fragments)
# answer = ask_deepseek(question, found)

По договору с «Альфа-Трейдинг» можем оплатить технику авансом или только после поставки?


In [4]:
import sys
from pathlib import Path

project = Path("..")
sys.path.insert(0, str(project))

from rag import load_and_split_documents, search, ask_deepseek

fragments = load_and_split_documents(project / "data" / "knowledge-base.zip")
#question = "По договору с «Альфа-Трейдинг» можем оплатить технику авансом или только после поставки?"
question = ques.iloc[8]["question"]
found = search(question, fragments)

for fragment in found:
    print(fragment.document, fragment.point)
    print(fragment.text[:250], end="\n\n")

Договор поставки ПТ-05-2023 Альфа-Трейдинг 1.1
1.1. Поставщик обязуется поставлять Покупателю компьютерную технику и комплектующие (далее — Товар) на основании заявок Покупателя, а Покупатель обязуется принимать и оплачивать Товар.

Договор поставки ПТ-05-2023 Альфа-Трейдинг 4.1
4.1. Покупатель оплачивает Товар в течение 45 (сорока пяти) календарных дней с даты поставки.

Договор поставки ПТ-05-2023 Альфа-Трейдинг 4.2
4.2. По соглашению Сторон допускается предоплата. Размер предоплаты указывается в счёте Поставщика.

Договор поставки ПТ-05-2023 Альфа-Трейдинг 5.2
5.2. За просрочку оплаты Покупатель уплачивает неустойку в размере 0,05% от неоплаченной суммы за каждый день просрочки.

Доп соглашение 1 к договору П-17-2024 
ДОПОЛНИТЕЛЬНОЕ СОГЛАШЕНИЕ № 1
к договору поставки № П-17/2024 от 12 февраля 2024 г.
г. Москва — 3 сентября 2024 г.
ООО «Альфа-Трейд» («Поставщик») в лице генерального директора Смирнова Олега Викторовича и ООО «Меридиан» («Покупатель») в лице генерал

Договор поставки 

In [6]:
import os
from getpass import getpass

os.environ["LLM_BASE_URL"] = "https://ai.starimg.ru/v1"
os.environ["LLM_MODEL"] = "deepseek-v4.1-flash-1017"
os.environ["LLM_API_KEY"] = getpass("Ключ DeepSeek: ").strip()

In [ ]:
import os
from getpass import getpass

import httpx

key = getpass("Вставь ключ DeepSeek: ").strip()

response = httpx.get(
    "https://ai.starimg.ru/v1/user/balance",
    headers={"Authorization": f"Bearer {key}"},
    timeout=15,
)

print("HTTP", response.status_code)

if response.status_code == 200:
    os.environ["LLM_API_KEY"] = key
    print("Ключ принят. Баланс доступен:", response.json()["is_available"])

In [ ]:
response = httpx.get(
    "https://ai.starimg.ru/v1/models",
    headers={"Authorization": f'Bearer {os.environ["LLM_API_KEY"]}'},
    timeout=15,
)
response.raise_for_status()

for model in response.json()["data"]:
    if "deepseek" in model["id"].lower():
        print(model["id"])

In [ ]:
question = ques.loc[ques["id"] == 9, "question"].iloc[0]
found = search(question, fragments)
answer = ask_deepseek(question, found)

print(answer)

In [7]:
import pandas as pd
from rag import load_and_split_documents, search, ask_deepseek

fragments = load_and_split_documents(project / "data" / "knowledge-base.zip")

results = []

In [9]:


#for row in ques.itertuples(index=False):
for row in ques.loc[ques["id"] >= 18].itertuples(index=False):
    found = search(row.question, fragments)
    answer = ask_deepseek(row.question, found)

    results.append({"id": row.id, "answer": answer})
    print(f"Готов вопрос {row.id}")


Готов вопрос 18


In [10]:
answers = pd.DataFrame(results)

In [12]:
expected = pd.read_csv(project / "data" / "demo_output.csv")
expected = expected.rename(columns={"answer": "expected_answer"})

comparison = answers.merge(expected, on="id")
comparison.to_csv(project / "comparison.csv", index=False, encoding="utf-8")
comparison[["id", "answer", "expected_answer"]]

,id,answer,expected_answer
0,1,Арендная плата составляет **450 000 (четыреста...,Арендная плата по договору аренды № А-08/2024 ...
1,2,Максимальный срок испытания для главного бухга...,Для главных бухгалтеров и их заместителей срок...
2,3,По переданным фрагментам размер пособия по бер...,В базе знаний нет документов о пособиях по бер...
3,4,Предпосылка неверна: по договору аренды № А-08...,По договору аренды № А-08/2024 с ООО «Северный...
4,5,"Да, нужно: договор на 12 млн рублей превышает ...",Да. В августе 2024 года действовал Регламент с...
5,6,Проверить качество товара нужно в течение 5 (п...,По договору поставки № П-17/2024 с ООО «Альфа-...
6,7,"Да, должны: по п. 4.3 договора аренды № А-08/2...",Да. По договору аренды № А-08/2024 с ООО «Севе...
7,8,К претензии по договору поставки № П-17/2024 н...,По договору поставки № П-17/2024 с ООО «Альфа-...
8,9,"Аванс возможен, но только по соглашению сторон...",Предоплата по договору поставки № ПТ-05/2023 с...
9,10,Согласование запускает руководитель подразделе...,По действующему Регламенту согласования догово...


In [ ]:
question = ques.loc[ques["id"] == 18, "question"].iloc[0]
first = search(question, fragments)[0]
print(first.document, first.point)

Регламент согласования договоров ред 2025 3.2


In [15]:
import os
import httpx

response = httpx.get(
    "https://ai.starimg.ru/v1/models",
    headers={"Authorization": f'Bearer {os.environ["LLM_API_KEY"]}'},
    timeout=15,
)

print("HTTP:", response.status_code)
response.raise_for_status()

model_ids = [model["id"] for model in response.json()["data"]]

print("Всего моделей:", len(model_ids))
print("Эмбеддинги:", [
    model for model in model_ids
    if any(word in model.lower() for word in ("embed", "bge", "e5"))
    and "rerank" not in model.lower()
])
print("Реранкеры:", [model for model in model_ids if "rerank" in model.lower()])

headers = {"Authorization": f'Bearer {os.environ["LLM_API_KEY"]}'}
base_url = "https://ai.starimg.ru/v1"

tests = {
    "embeddings": {
        "model": "baai/bge-m3",
        "input": ["Можно ли перечислить деньги вперёд?"],
    },
    "rerank": {
        "model": "Qwen/Qwen3-Reranker-8B",
        "query": "Можно ли перечислить деньги вперёд?",
        "documents": ["Предоплата допускается по соглашению сторон."],
    },
}

for endpoint, payload in tests.items():
    response = httpx.post(
        f"{base_url}/{endpoint}", headers=headers, json=payload, timeout=20
    )
    print(f"/{endpoint}: HTTP {response.status_code}")
    if not response.is_success:
        print(response.text[:300].replace(os.environ["LLM_API_KEY"], "[скрыто]"))

HTTP: 200
Всего моделей: 48
Эмбеддинги: []
Реранкеры: []
/embeddings: HTTP 404
404 page not found

/rerank: HTTP 404
404 page not found



In [14]:
headers = {"Authorization": f'Bearer {os.environ["LLM_API_KEY"]}'}
base_url = "https://ai.starimg.ru/v1"

tests = {
    "embeddings": {
        "model": "baai/bge-m3",
        "input": ["Можно ли перечислить деньги вперёд?"],
    },
    "rerank": {
        "model": "Qwen/Qwen3-Reranker-8B",
        "query": "Можно ли перечислить деньги вперёд?",
        "documents": ["Предоплата допускается по соглашению сторон."],
    },
}

for endpoint, payload in tests.items():
    response = httpx.post(
        f"{base_url}/{endpoint}", headers=headers, json=payload, timeout=20
    )
    print(f"/{endpoint}: HTTP {response.status_code}")
    if not response.is_success:
        print(response.text[:300].replace(os.environ["LLM_API_KEY"], "[скрыто]"))

/embeddings: HTTP 404
404 page not found

/rerank: HTTP 404
404 page not found



In [16]:
import math
import httpx

candidates = [
    f for f in fragments
    if "ПТ-05-2023" in f.document and f.point in {"1.1", "4.1", "4.2", "5.2"}
]

response = httpx.post(
    "http://127.0.0.1:11434/api/embed",
    json={
        "model": "bge-m3",
        "input": ["Можно перечислить деньги вперёд?"] + [f.text for f in candidates],
    },
    timeout=120,
)
response.raise_for_status()

query_vector, *vectors = response.json()["embeddings"]

def similarity(a, b):
    return sum(x * y for x, y in zip(a, b)) / (math.hypot(*a) * math.hypot(*b))

ranked = sorted(
    zip(candidates, vectors),
    key=lambda pair: similarity(query_vector, pair[1]),
    reverse=True,
)

for fragment, vector in ranked:
    print(f"{similarity(query_vector, vector):.3f} | п. {fragment.point} | {fragment.text}")

0.652 | п. 4.2 | 4.2. По соглашению Сторон допускается предоплата. Размер предоплаты указывается в счёте Поставщика.
0.465 | п. 4.1 | 4.1. Покупатель оплачивает Товар в течение 45 (сорока пяти) календарных дней с даты поставки.
0.433 | п. 5.2 | 5.2. За просрочку оплаты Покупатель уплачивает неустойку в размере 0,05% от неоплаченной суммы за каждый день просрочки.
0.430 | п. 1.1 | 1.1. Поставщик обязуется поставлять Покупателю компьютерную технику и комплектующие (далее — Товар) на основании заявок Покупателя, а Покупатель обязуется принимать и оплачивать Товар.


In [17]:
def embed(texts):
    response = httpx.post(
        "http://127.0.0.1:11434/api/embed",
        json={"model": "bge-m3", "input": texts},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["embeddings"]

# Считаем один раз и переиспользуем для следующих вопросов.
fragment_vectors = []
batch_size = 16

for start in range(0, len(fragments), batch_size):
    batch = fragments[start:start + batch_size]
    fragment_vectors.extend(embed([f"{f.document}\n{f.text}" for f in batch]))
    print(f"Готово {len(fragment_vectors)}/{len(fragments)}")

question = "Можно перечислить деньги вперёд поставщику компьютеров по договору ПТ-05/2023?"
query_vector = embed([question])[0]

ranked = sorted(
    zip(fragments, fragment_vectors),
    key=lambda pair: similarity(query_vector, pair[1]),
    reverse=True,
)

for rank, (fragment, vector) in enumerate(ranked[:10], start=1):
    print(rank, round(similarity(query_vector, vector), 3), fragment.document, fragment.point)

Готово 16/409
Готово 32/409
Готово 48/409
Готово 64/409
Готово 80/409
Готово 96/409
Готово 112/409
Готово 128/409
Готово 144/409
Готово 160/409
Готово 176/409
Готово 192/409
Готово 208/409
Готово 224/409
Готово 240/409
Готово 256/409
Готово 272/409
Готово 288/409
Готово 304/409
Готово 320/409
Готово 336/409
Готово 352/409
Готово 368/409
Готово 384/409
Готово 400/409
Готово 409/409
1 0.717 Договор поставки ПТ-05-2023 Альфа-Трейдинг 4.2
2 0.667 Договор поставки ПТ-05-2023 Альфа-Трейдинг 1.1
3 0.625 Договор поставки ПТ-05-2023 Альфа-Трейдинг 4.1
4 0.623 Договор поставки ПТ-05-2023 Альфа-Трейдинг 4
5 0.616 Договор поставки П-17-2024 Альфа-Трейд 4.1
6 0.615 Договор поставки ПТ-05-2023 Альфа-Трейдинг 
7 0.613 Договор поставки ПТ-05-2023 Альфа-Трейдинг 3
8 0.6 Договор поставки П-17-2024 Альфа-Трейд 4.2
9 0.599 Договор поставки П-17-2024 Альфа-Трейд 1.1
10 0.595 Договор поставки ПТ-05-2023 Альфа-Трейдинг 3.1


In [18]:
import pandas as pd

tests = [
    ("Одобрение в 2024", "Гендиректор должен был одобрить сделку на 12 миллионов в августе 2024 года?",
     [("ред 2023", "2.4")]),
    ("Ночной проезд", "Может ли охрана отказать в проезде автомобилю арендатора в 02:00 по договору с Северным Складом?",
     [("Северный Склад", "4.3")]),
    ("Инициатор", "Кто у нас начинает процедуру утверждения проекта договора?",
     [("ред 2025", "1.3")]),
    ("Оригинал", "Когда после заключения сделки надо сдать бумажный экземпляр договора в юридический отдел?",
     [("ред 2025", "3.2")]),
    ("Предоплата", "Можно перечислить деньги вперёд поставщику компьютеров по договору ПТ-05/2023?",
     [("ПТ-05-2023", "4.2"), ("ПТ-05-2023", "4.1")]),
    ("Приёмка", "По П-17/2024 когда заканчивается срок осмотра товара на дефекты после доставки?",
     [("П-17-2024 Альфа-Трейд", "6.1"), ("П-17-2024 Альфа-Трейд", "3.3")]),
    ("Отзыв согласия", "Мы отозвали согласие клиента. Допускается ли дальше хранить его данные?",
     [("152ФЗ_ст9", "2")]),
    ("Возврат склада", "Покидаем склад по А-08/2024. Нужно ли подписывать бумагу о возврате помещения?",
     [("Северный Склад", "6.3")]),
    ("Смена адреса", "Куда доставлять мебель по договору П-17/2024, если спецификацию подписали 15.10.2024?",
     [("Доп соглашение", "")]),
    ("Инициатор — прямой вопрос", "Кто инициирует согласование проекта нового договора?",
     [("ред 2025", "1.3")]),
    ("Аванс — прямой вопрос", "Допускается ли аванс по договору ПТ-05/2023 и когда оплатить товар без аванса?",
     [("ПТ-05-2023", "4.2"), ("ПТ-05-2023", "4.1")]),
]


def find_ranks(found, required):
    return [
        next(
            (rank for rank, fragment in enumerate(found, start=1)
             if document in fragment.document and fragment.point == point),
            None,
        )
        for document, point in required
    ]


query_vectors = embed([question for _, question, _ in tests])
results = []

for (name, question, required), query_vector in zip(tests, query_vectors, strict=True):
    by_words = search(question, fragments, top_k=6)
    ranked = sorted(
        zip(fragments, fragment_vectors, strict=True),
        key=lambda pair: similarity(query_vector, pair[1]),
        reverse=True,
    )
    by_meaning = [fragment for fragment, _ in ranked[:6]]

    word_ranks = find_ranks(by_words, required)
    meaning_ranks = find_ranks(by_meaning, required)

    results.append({
        "вопрос": name,
        "по словам": word_ranks,
        "по смыслу": meaning_ranks,
        "все пункты по словам": None not in word_ranks,
        "все пункты по смыслу": None not in meaning_ranks,
    })

report = pd.DataFrame(results)

print("Все нужные пункты в топ-6:")
print(f"По словам: {report['все пункты по словам'].sum()} из {len(report)}")
print(f"По смыслу: {report['все пункты по смыслу'].sum()} из {len(report)}")

report[["вопрос", "по словам", "по смыслу"]]

Все нужные пункты в топ-6:
По словам: 6 из 11
По смыслу: 10 из 11


,вопрос,по словам,по смыслу
0,Одобрение в 2024,[4],[1]
1,Ночной проезд,[None],[1]
2,Инициатор,[None],[3]
3,Оригинал,[None],[1]
4,Предоплата,"[None, None]","[1, 3]"
5,Приёмка,"[None, None]","[3, None]"
6,Отзыв согласия,[1],[1]
7,Возврат склада,[1],[3]
8,Смена адреса,[1],[1]
9,Инициатор — прямой вопрос,[1],[1]


In [19]:
import os
from importlib import reload

import pandas as pd
import rag

reload(rag)

os.environ["EMBEDDING_BASE_URL"] = "http://127.0.0.1:11434/v1"
os.environ["EMB_MODEL"] = "bge-m3"

index = rag.build_embedding_index(fragments, vectors=fragment_vectors)
assert index is not None, "Не удалось подключить эмбеддинги"

rows = []
for name, question, required in tests:
    found = rag.search(question, fragments, embedding_index=index)
    answer = rag.ask_deepseek(question, found)

    rows.append({
        "test": name,
        "question": question,
        "expected_sources": str(required),
        "answer": answer,
        "context": "\n\n".join(
            f"{f.document}, п. {f.point}\n{f.text}" for f in found
        ),
    })
    print(f"Готово: {name}")

report = pd.DataFrame(rows)
report.to_csv(project / "comparison_v2_variants.csv", index=False)

Готово: Одобрение в 2024
Готово: Ночной проезд
Готово: Инициатор
Готово: Оригинал
Готово: Предоплата
Готово: Приёмка
Готово: Отзыв согласия
Готово: Возврат склада
Готово: Смена адреса
Готово: Инициатор — прямой вопрос
Готово: Аванс — прямой вопрос


In [20]:
import pandas as pd
from holdout_cases import run_holdout

report = pd.DataFrame(run_holdout(fragments, index))
report.to_csv(project / "comparison_v2_holdout.csv", index=False)

report[["test", "expected", "answer", "missing_sources"]]

Готово: Смягчение налогового штрафа
Готово: Повторное налоговое нарушение
Готово: Два налоговых нарушения
Готово: Историческая редакция НК
Готово: Граница редакций регламента
Готово: Граница смены адреса
Готово: Неоднозначный поставщик
Готово: Неустойка за задержку поставки
Готово: Кто отзывает согласие
Готово: Сверхурочные после 120 часов
Готово: Госпошлина вне базы


,test,expected,answer,missing_sources
0,Смягчение налогового штрафа,"Уменьшение от 2 до 10 раз, то есть итоговый шт...",Штраф 100 000 рублей при одном смягчающем обст...,
1,Повторное налоговое нарушение,Аналогичное нарушение в пределах 12 месяцев яв...,Уточнить изменение штрафа невозможно: в переда...,"НК_ст112, п. 2\nНК_ст112, п. 3\nНК_ст114, п. 4"
2,Два налоговых нарушения,Нет. Налоговая санкция взыскивается за каждое ...,"Нет, нельзя: при совершении одним лицом двух и...",
3,Историческая редакция НК,Подтвердить нельзя. Доступный п. 3 ст. 114 НК ...,Подтвердить «предел снижения ровно в десять ра...,
4,Граница редакций регламента,"Для первого требовалось, поскольку порог по ре...",Одобрение гендиректора требовалось по первому ...,
5,Граница смены адреса,"На 01.10 действует исходный адрес: г. Москва, ...",Партию по спецификации от 01.10.2024 везите по...,
6,Неоднозначный поставщик,Уточнить поставщика или номер договора. Допуст...,"Уточните, пожалуйста, с каким именно поставщик...",
7,Неустойка за задержку поставки,"0,1% от стоимости товара, не поставленного в с...",Неустойка по п. 7.1 договора П-17/2024 рассчит...,
8,Кто отзывает согласие,Часть 2 предусматривает отзыв самим субъектом ...,"Нет, по ч. 2 ст. 9 152-ФЗ отозвать согласие мо...",
9,Сверхурочные после 120 часов,Начиная со 121-го часа за год оплата не менее ...,Минимальная оплата — не менее чем в двойном ра...,


In [21]:
import os
from importlib import reload

import rag
from holdout_cases import CASES

reload(rag)

os.environ["EMBEDDING_BASE_URL"] = "http://127.0.0.1:11434/v1"
os.environ["EMB_MODEL"] = "bge-m3"

# Заново читаем документы, чтобы получить ссылки с новой нумерацией.
fragments = rag.load_and_split_documents(project / "data" / "knowledge-base.zip")
index = rag.build_embedding_index(fragments, vectors=fragment_vectors)
assert index is not None, "Не удалось подключить индекс BGE-M3"

# Первые два вопроса — смягчение и повторное налоговое нарушение.
for name, question, expected, _ in CASES[:2]:
    found = rag.search(question, fragments, embedding_index=index)

    print(f"\n{name}")
    print(question)
    print("\nИсточники:")
    for fragment in found:
        print("-", fragment.reference)

    print("\nОтвет DeepSeek:", rag.ask_deepseek(question, found))
    print("Ожидаемый вывод:", expected)


Смягчение налогового штрафа
В октябре 2026 года налоговая признала одно смягчающее обстоятельство. Штраф по статье — 100 000 рублей. В каких пределах его могут уменьшить?

Источники:
- п. 3 ст. 114 НК РФ
- ст. 112 НК РФ
- п. 1 ст. 112 НК РФ
- п. 4 ст. 114 НК РФ
- п. 2 ст. 112 НК РФ
- подп. 3 п. 1 ст. 112 НК РФ

Ответ DeepSeek: Штраф могут уменьшить не менее чем в два раза и не более чем в десять раз по сравнению со 100 000 рублей, то есть в пределах от 10 000 до 50 000 рублей (п. 3 ст. 114 НК РФ; на октябрь 2026 года действует редакция с изменениями по Федеральному закону от 28 ноября 2025 г. № 425-ФЗ).
Ожидаемый вывод: Уменьшение от 2 до 10 раз, то есть итоговый штраф от 10 000 до 50 000 рублей. НК РФ, ст. 114, п. 3.

Повторное налоговое нарушение
Компания совершила такое же налоговое нарушение через восемь месяцев после вступления в силу решения, по которому с неё взыскали штраф. Как изменится новый штраф в 40 000 рублей?

Источники:
- п. 3 ст. 112 НК РФ
- п. 2 ст. 114 НК РФ
- п. 2 